# **PREPARATION**

Environment: see `requirements.txt` in the repository root. Run from the `notebooks/` directory.


In [ ]:
import os
import random
import hashlib
import pickle
from itertools import combinations, product

import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

DATA_DIR = "../data"


In [ ]:
df = pd.read_csv(f'{DATA_DIR}/data_train.csv')

In [ ]:
emb_cols = [
    c
    for c in df.columns
    if c.startswith("chemberta_")
]

X_emb = df[emb_cols].values

print(X_emb.shape)

(834, 768)


In [ ]:
library = pd.read_csv(f'{DATA_DIR}/virtual_library.csv')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [ ]:
MODEL_NAME = "DeepChem/ChemBERTa-100M-MLM"
MODEL_REVISION = "f5c45f44d3061f0346888f5c09db17ec1146d29"  # pinned HF revision (see results/screening_run/run_metadata_*.txt)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)

model = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)

model.to(device)

model.eval();

config.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/149k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/101k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/384k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-100M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    return (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

In [ ]:
@torch.no_grad()
def get_embedding(smiles):

    encoded = tokenizer(
        smiles,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    encoded = {
        k: v.to(device)
        for k, v in encoded.items()
    }

    output = model(**encoded)

    emb = mean_pooling(
        output,
        encoded["attention_mask"]
    )

    return emb.squeeze().cpu().numpy()

In [ ]:
library_smiles = library["SMILES_nowater"].tolist()

In [ ]:
emb = get_embedding(library.iloc[0]["SMILES_nowater"])

print(emb.shape)

(768,)


In [ ]:
@torch.no_grad()
def get_embeddings_batch(smiles_list):

    encoded = tokenizer(
        smiles_list,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    encoded = {
        k: v.to(device)
        for k, v in encoded.items()
    }

    output = model(**encoded)

    emb = mean_pooling(
        output,
        encoded["attention_mask"]
    )

    return emb.cpu().numpy()

In [ ]:
BATCH_SIZE = 1

smiles = library["SMILES_nowater"].tolist()

embeddings = []

for i in tqdm(range(0, len(smiles), BATCH_SIZE)):

    batch = smiles[i:i+BATCH_SIZE]

    emb = get_embeddings_batch(batch)

    embeddings.append(emb)

embeddings = np.vstack(embeddings)

print(embeddings.shape)

  0%|          | 0/20 [00:00<?, ?it/s]

(20, 768)


In [ ]:
X_emb_library = embeddings

print(X_emb_library.shape)

(20, 768)


In [ ]:
groups = df["metal_group"].values

# **TRAINING**

In [ ]:
library

,Unnamed: 0,ID,Status,Fe_oxidation_state,Family,Second_metal,Counterion,Gross_formula,Ligand_formula,SMILES_complex,SMILES_nowater
0,0,P01,Virtual Candidate,III,oxalate,-,NH4+,C6H18FeN3O15,3xC2O4^-2 + 3xH2O,[NH4+].[NH4+].[NH4+].[Fe+3].[O-]C(=O)C(=O)[O-]...,[NH4+].[NH4+].[NH4+].[Fe+3].[O-]C(=O)C(=O)[O-]...
1,1,P02,Virtual Candidate,II,fumarate,Zn(II),-,C8H12FeO12Zn,2xC4H2O4^-2 + 4xH2O,[Zn+2].[Fe+2].[O-]C(=O)/C=C/C(=O)[O-].[O-]C(=O...,[Zn+2].[Fe+2].[O-]C(=O)/C=C/C(=O)[O-].[O-]C(=O...
2,2,P03,Virtual Candidate,III,oxalate,Ba(II),NH4+,C6H12BaFeNO16,3xC2O4^-2 + 4xH2O,[NH4+].[Ba+2].[Fe+3].[O-]C(=O)C(=O)[O-].[O-]C(...,[NH4+].[Ba+2].[Fe+3].[O-]C(=O)C(=O)[O-].[O-]C(...
3,3,P04,Virtual Candidate,III,succinate,Ba(II),NH4+,C12H24BaFeNO16,3xC4H4O4^-2 + 4xH2O,[NH4+].[Ba+2].[Fe+3].[O-]C(=O)CCC(=O)[O-].[O-]...,[NH4+].[Ba+2].[Fe+3].[O-]C(=O)CCC(=O)[O-].[O-]...
4,4,P05,Virtual Candidate,III,malonate,Ba(II),Et4N+,C17H34BaFeNO16,3xC3H2O4^-2 + 4xH2O,CC[N+](CC)(CC)CC.[Ba+2].[Fe+3].[O-]C(=O)CC(=O)...,CC[N+](CC)(CC)CC.[Ba+2].[Fe+3].[O-]C(=O)CC(=O)...
5,5,P06,Virtual Candidate,III,oxalate,Cu(II),NH4+,C6H4CuFeNO12,3xC2O4^-2,[NH4+].[Cu+2].[Fe+3].[O-]C(=O)C(=O)[O-].[O-]C(...,[NH4+].[Cu+2].[Fe+3].[O-]C(=O)C(=O)[O-].[O-]C(...
6,6,P07,Virtual Candidate,III,oxalate,Mn(II),Et4N+,C14H20FeMnNO12,3xC2O4^-2,CC[N+](CC)(CC)CC.[Mn+2].[Fe+3].[O-]C(=O)C(=O)[...,CC[N+](CC)(CC)CC.[Mn+2].[Fe+3].[O-]C(=O)C(=O)[...
7,7,P08,Virtual Candidate,III,oxalate,Co(II),NH4+,C6H4CoFeNO12,3xC2O4^-2,[NH4+].[Co+2].[Fe+3].[O-]C(=O)C(=O)[O-].[O-]C(...,[NH4+].[Co+2].[Fe+3].[O-]C(=O)C(=O)[O-].[O-]C(...
8,8,P09,Experimental Prototype,III,malonate,-,Et4N+,C14H28FeNO10,2xC3H2O4^-2 + 2xH2O,CC[N+](CC)(CC)CC.[Fe+3].[O-]C(=O)CC(=O)[O-].[O...,CC[N+](CC)(CC)CC.[Fe+3].[O-]C(=O)CC(=O)[O-].[O...
9,9,P10,Virtual Candidate,II,oxalate,-,NH4+,C4H12FeN2O10,2xC2O4^-2 + 2xH2O,[NH4+].[NH4+].[Fe+2].[O-]C(=O)C(=O)[O-].[O-]C(...,[NH4+].[NH4+].[Fe+2].[O-]C(=O)C(=O)[O-].[O-]C(...


In [ ]:
param_grid = list(product(

    [90, 95, 100, 105, 110],          # n_estimators

    ["auto", 218, 243, 269],      # max_samples

    [0.75, 0.80, 0.85, 1.0],             # max_features

))

print(f"Total models: {len(param_grid)}")

Total models: 80


In [ ]:
screening_scores = library[["ID", "Gross_formula"]].copy()

for (

    n_estimators,

    max_samples,

    max_features,

) in tqdm(param_grid):

    pipe = Pipeline([

        ("imputer", SimpleImputer(strategy="median")),

        ("scaler", StandardScaler()),

        ("model",

            IsolationForest(

                n_estimators=n_estimators,

                contamination="auto",

                random_state=42,

                max_samples=max_samples,

                max_features=max_features,

            )

        )

    ])

    pipe.fit(X_emb)

    scores = pipe.decision_function(X_emb_library)

    col = (

        f"N{n_estimators}"

        f"_MS{max_samples}"

        f"_MF{max_features}"

    )

    screening_scores[col] = scores

print(screening_scores.shape)

  0%|          | 0/80 [00:00<?, ?it/s]

(20, 82)


In [ ]:
rank_table = screening_scores.copy()

score_cols = rank_table.columns[2:]

for c in score_cols:

    rank_table[c] = (

        rank_table[c]

        .rank(

            ascending=False,

            method="min"

        )

        .astype(int)

    )

display(rank_table)

,ID,Gross_formula,N90_MSauto_MF0.75,N90_MSauto_MF0.8,N90_MSauto_MF0.85,N90_MSauto_MF1.0,N90_MS218_MF0.75,N90_MS218_MF0.8,N90_MS218_MF0.85,N90_MS218_MF1.0,...,N110_MS218_MF0.85,N110_MS218_MF1.0,N110_MS243_MF0.75,N110_MS243_MF0.8,N110_MS243_MF0.85,N110_MS243_MF1.0,N110_MS269_MF0.75,N110_MS269_MF0.8,N110_MS269_MF0.85,N110_MS269_MF1.0
0,P01,C6H18FeN3O15,17,20,20,20,20,19,20,20,...,20,20,20,20,20,20,19,20,20,20
1,P02,C8H12FeO12Zn,19,8,8,2,15,2,8,1,...,11,1,15,7,7,1,10,3,3,4
2,P03,C6H12BaFeNO16,7,14,12,11,11,10,9,9,...,9,14,7,14,11,13,15,18,12,14
3,P04,C12H24BaFeNO16,12,2,3,14,18,6,5,6,...,7,8,10,5,9,9,14,7,13,5
4,P05,C17H34BaFeNO16,8,9,7,18,4,18,15,17,...,14,18,2,6,10,18,3,6,9,12
5,P06,C6H4CuFeNO12,6,15,12,12,10,12,12,9,...,10,14,11,15,12,15,16,15,15,15
6,P07,C14H20FeMnNO12,5,3,4,9,2,14,4,5,...,4,3,3,2,4,12,2,1,7,9
7,P08,C6H4CoFeNO12,1,16,16,15,8,13,16,14,...,16,16,5,16,14,16,13,17,17,16
8,P09,C14H28FeNO10,11,11,17,6,17,5,3,11,...,5,9,1,3,13,5,8,8,11,6
9,P10,C4H12FeN2O10,20,13,1,4,3,7,1,4,...,2,4,14,10,2,6,9,4,2,3


In [ ]:
summary = pd.DataFrame()

summary["ID"] = rank_table["ID"]

summary["Gross_formula"] = rank_table["Gross_formula"]

summary = pd.DataFrame()
summary["ID"] = rank_table["ID"]
summary["Gross_formula"] = rank_table["Gross_formula"]

score_data = rank_table.iloc[:, 2:]

summary["Mean_rank"] = score_data.mean(axis=1)
summary["SD_rank"] = score_data.std(axis=1)
summary["Top10_%"] = (score_data <= 10).mean(axis=1) * 100
summary["Top5_%"]  = (score_data <= 5).mean(axis=1) * 100
summary["Top3_%"]  = (score_data <= 3).mean(axis=1) * 100
summary["Top2_%"]  = (score_data <= 2).mean(axis=1) * 100
summary["Rank1_%"] = (score_data == 1).mean(axis=1) * 100

summary = summary.sort_values("Mean_rank")
display(summary)


,ID,Gross_formula,Mean_rank,SD_rank,Top10_%,Top5_%,Top3_%,Top2_%,Rank1_%
6,P07,C14H20FeMnNO12,5.0625,3.766420,87.50,68.75,36.25,30.00,12.50
18,P19,C12H24CuFe2O16,5.4000,4.718936,76.25,58.75,55.00,50.00,27.50
16,P17,C2H14CoFeO10P2,6.2875,4.492451,80.00,50.00,31.25,30.00,15.00
9,P10,C4H12FeN2O10,6.3125,4.748334,80.00,53.75,38.75,18.75,11.25
14,P15,C3H13FeNO9P3Zn,6.7500,3.570271,83.75,51.25,23.75,6.25,0.00
12,P13,C9H22BaFeNO18,7.0000,3.162278,86.25,36.25,18.75,7.50,1.25
1,P02,C8H12FeO12Zn,7.2250,5.317692,73.75,41.25,32.50,27.50,18.75
8,P09,C14H28FeNO10,8.5875,4.462768,67.50,28.75,11.25,8.75,6.25
3,P04,C12H24BaFeNO16,8.6625,4.106012,66.25,25.00,8.75,3.75,2.50
19,P20,C8H12FeMnNO14,9.5250,4.012559,57.50,20.00,12.50,2.50,0.00


# **SAVE**

In [ ]:
PROJECT_DIR = '../results/screening_run'
os.makedirs(PROJECT_DIR, exist_ok=True)

In [ ]:
import torch, sklearn, numpy, transformers, pandas, sys, subprocess

env_info = f"""
Date screening: {pd.Timestamp.now()}
Python: {sys.version}
torch: {torch.__version__}
sklearn: {sklearn.__version__}
numpy: {numpy.__version__}
transformers: {transformers.__version__}
pandas: {pandas.__version__}
CUDA available: {torch.cuda.is_available()}
"""



with open(f"{PROJECT_DIR}/environment_log.txt", "a") as f:
    f.write(env_info + "\n" + "="*50 + "\n")

print(env_info)


Date scrinig: 2026-08-09 18:03:32.226280
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.11.0+cpu
sklearn: 1.6.1
numpy: 2.0.2
transformers: 5.13.1
pandas: 2.2.2
CUDA available: False



In [ ]:
MODEL_NAME = "DeepChem/ChemBERTa-100M-MLM"
MODEL_REVISION = "f5c45f44d3061f0346888f5c09db17ec1146d29"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-100M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import pickle

EMB_CACHE_PATH = f'{PROJECT_DIR}/embeddings_cache.pkl'

def get_embeddings_cached(smiles_list, cache_path=EMB_CACHE_PATH):
    cache = {}
    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)

    to_compute = [s for s in smiles_list if s not in cache]
    if to_compute:
        print(f"Computing {len(to_compute)} new embeddings (out of {len(smiles_list)})")
        new_embs = get_embeddings_batch(to_compute)
        for s, e in zip(to_compute, new_embs):
            cache[s] = e
        with open(cache_path, "wb") as f:
            pickle.dump(cache, f)
    else:
        print("All embeddings loaded from cache — model was not run")

    return np.vstack([cache[s] for s in smiles_list])

In [ ]:
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import hashlib

def dataset_hash(df):
    return hashlib.md5(pd.util.hash_pandas_object(df).values).hexdigest()

print("library hash:", dataset_hash(library[["ID", "SMILES_nowater"]]))

library hash: 29b7dabf18252265bb49ef48cb509227


In [ ]:
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
summary.to_csv(f"{PROJECT_DIR}/summary_{timestamp}.csv", index=False)

with open(f"{PROJECT_DIR}/run_metadata_{timestamp}.txt", "w") as f:
    f.write(env_info)
    f.write(f"\nDataset hash: {dataset_hash(library[['ID','SMILES_nowater']])}\n")
    f.write(f"Model revision: {MODEL_REVISION}\n")